# 05. Skill Optimization — dialog-summary SKILL.md
Optimize `dialog-summary/SKILL.md` against samsum gold summaries via Arize experiments.

In [1]:
!pip install -qqq arize arize-phoenix certifi urllib3 langchain-openai python-dotenv

In [2]:
import os, certifi
CA = certifi.where()
os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

from arize import ArizeClient
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set (not in .env) — export it before running"

client = ArizeClient(api_key=API_KEY, request_verify=False)

# ---- demo config (single source of truth) ----
N_EXAMPLES   = 50
N_ITERATIONS = 3
GEN_MODEL    = "gpt-4.1-2025-04-14"
JUDGE_MODEL  = "gpt-4o-mini"
SKILL_PATH   = "dialog-summary/SKILL.md"
VERSIONS_DIR = "dialog-summary/versions"
DATASET_NAME = "samsum_small"

In [3]:
examples = client.datasets.list_examples(dataset=DATASET_NAME, space=SPACE_ID)
examples

  arize.pre_releases | WARNING | [BETA] datasets.list_examples is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


DatasetExampleListResponse(examples=[DatasetExample(id='00f72bad-ab78-4d51-9df7-e4be79905611', created_at=datetime.datetime(2026, 6, 26, 3, 25, 16, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 26, 3, 25, 16, tzinfo=TzInfo(0)), annotations=None, additional_properties={'dialogue': "Mary: Sorry, I didn't make it to your bday party :(\nNick: It's OK...\nMary: But I just got SOOO distracted! I forgot it was yesterday!\nNick: do tell!\nMary: I met this guy...\nNick: REALLY? I want details :D\nMary: Yeah, his name is Kirk and he's an architect...\nNick: OK, just your type then <file_gif>\nMary: And we ended up spending the whole week together. xD\nNick: A WEEK?\nMary: Yeah... It's madness, I'll tell you more this evening. Are we still on?\nNick: You bet we are!", 'summary': "Mary didn't come to Nick's birthday party. She met an architect named Kirk. Mary and Nick will meet in the evening."}), DatasetExample(id='02a40690-2e21-403c-b883-26e1b263e4d9', created_at=datetime.datetime(20

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

gen_llm = ChatOpenAI(model=GEN_MODEL, temperature=0.0)
gen_chain = gen_llm | StrOutputParser()

def read_skill() -> str:
    with open(SKILL_PATH, "r") as f:
        return f.read()

def run_skill(dataset_row) -> str:
    """Use the full SKILL.md text as the system prompt over one dialogue."""
    skill = read_skill()
    dialogue = dataset_row["dialogue"]
    messages = [
        ("system", skill),
        ("human", f"Summarize this dialogue:\n\n{dialogue}"),
    ]
    return gen_chain.invoke(messages)

In [5]:
# smoke test on one example (does NOT hit Arize)
_sample = examples.to_df().iloc[0] if hasattr(examples, "to_df") else pd.DataFrame(examples).iloc[0]
print("DIALOGUE:\n", _sample["dialogue"][:400])
print("\nGOLD:\n", _sample["summary"])
print("\nSKILL OUTPUT:\n", run_skill(_sample))

DIALOGUE:
 Mary: Sorry, I didn't make it to your bday party :(
Nick: It's OK...
Mary: But I just got SOOO distracted! I forgot it was yesterday!
Nick: do tell!
Mary: I met this guy...
Nick: REALLY? I want details :D
Mary: Yeah, his name is Kirk and he's an architect...
Nick: OK, just your type then <file_gif>
Mary: And we ended up spending the whole week together. xD
Nick: A WEEK?
Mary: Yeah... It's madness,

GOLD:
 Mary didn't come to Nick's birthday party. She met an architect named Kirk. Mary and Nick will meet in the evening.



SKILL OUTPUT:
 Mary apologizes to Nick for missing his birthday party because she was distracted after meeting someone new, Kirk, with whom she spent the entire week. She promises to share more details when they meet later that evening, and Nick confirms their plans.


In [6]:
from phoenix.evals import llm_classify, OpenAIModel
from arize.experiments import EvaluationResult

skill_eval_template = """
You are judging whether a generated summary of a dialogue matches the reference (gold) summary.
    [BEGIN DATA]
    ************
    [Generated Summary]: {output}
    ************
    [Reference Summary]: {reference}
    [END DATA]

Compare the Generated Summary to the Reference Summary. Judge whether the Generated Summary is
faithful (no invented facts, correct who-said/wanted-what), covers the same key decisions/requests/
plans as the Reference, and is comparably concise. Your response must be a single word, either
"good" or "bad", with no other text. "good" = faithful, complete on key points, and concise
relative to the Reference. "bad" = misses key points, misattributes, invents facts, or is not concise.
"""

def skill_eval(output, dataset_row) -> EvaluationResult:
    eval_df = llm_classify(
        dataframe=pd.DataFrame([{"output": output, "reference": dataset_row["summary"]}]),
        template=skill_eval_template,
        model=OpenAIModel(model=JUDGE_MODEL),
        rails=["good", "bad"],
        provide_explanation=True,
    )
    label = eval_df["label"][0]
    score = 1 if label == "good" else 0
    return EvaluationResult(label=label, score=score, explanation=eval_df["explanation"][0])

/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# sanity: a good summary (the gold itself) should score 1; a junk summary should score 0
print("gold-vs-gold:", skill_eval(_sample["summary"], _sample).label, "(expect good)")
print("junk        :", skill_eval("They talked about nothing.", _sample).label, "(expect bad)")

/var/folders/3n/gv0dncrs175g36_29rs1d_r00000gn/T/ipykernel_7591/1575363867.py:21: DeprecationWarning: `dataframe` argument is deprecated; use `data` instead
  eval_df = llm_classify(
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:04<00:00 |  4.61s/it

llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:04<00:00 |  4.61s/it


🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


gold-vs-gold: good (expect good)


llm_classify |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:02<00:00 |  2.35s/it

llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:02<00:00 |  2.35s/it

junk        : bad (expect bad)


## Task 4: Baseline experiment `skill-v0` + feedback collection

Run the first Arize experiment (`task=run_skill`, `evaluators=[skill_eval]`) over `samsum_small`, then collect a small mixed sample of "bad"/"good" judged rows to feed the optimizer in later tasks.

In [8]:
def _delete_experiment_if_exists(version_name: str):
    """Idempotency: drop any prior experiment with this name so the notebook is re-runnable
    (Arize rejects experiments.run when the name already exists on the dataset)."""
    existing = client.experiments.list(space=SPACE_ID, dataset=DATASET_NAME)
    for e in existing.experiments:
        if e.name == version_name:
            client.experiments.delete(experiment=e.id, space=SPACE_ID, dataset=DATASET_NAME)

def run_experiment(version_name: str):
    _delete_experiment_if_exists(version_name)
    # experiments.run returns (experiment, results_df); we use the returned df directly.
    # NOTE: client.experiments.list_runs(...).to_df() is buggy in arize SDK v8.37.1
    # (raises pydantic ValidationError: ExperimentRun.id is None), so we avoid it.
    experiment, eval_df = client.experiments.run(
        name=version_name,
        dataset=DATASET_NAME,
        space=SPACE_ID,
        task=run_skill,
        evaluators=[skill_eval],
    )
    return experiment, eval_df

def mean_score(eval_df) -> float:
    return float(eval_df["eval.skill_eval.score"].astype(float).mean())

def collect_feedback(eval_df, n_bad: int = 3, n_good: int = 5):
    bad  = eval_df[eval_df["eval.skill_eval.label"] == "bad"].head(n_bad)
    good = eval_df[eval_df["eval.skill_eval.label"] == "good"].head(n_good)
    return pd.concat([bad, good])

In [9]:
exp_v0, eval_v0 = run_experiment("skill-v0")
print("columns:", list(eval_v0.columns))
print("skill-v0 mean score:", mean_score(eval_v0))
eval_v0.head()

  arize.pre_releases | WARNING | [BETA] experiments.list is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  arize.pre_releases | WARNING | [BETA] experiments.delete is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

  arize.experiments.functions | INFO | ✅ All evaluators completed.


  arize.pre_releases | WARNING | [BETA] experiments.get is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


columns: ['example_id', 'output', 'error', 'result.trace.id', 'result.trace.timestamp', 'eval.skill_eval.score', 'eval.skill_eval.label', 'eval.skill_eval.explanation', 'eval.skill_eval.trace.id', 'eval.skill_eval.trace.timestamp']
skill-v0 mean score: 0.29


,example_id,output,error,result.trace.id,result.trace.timestamp,eval.skill_eval.score,eval.skill_eval.label,eval.skill_eval.explanation,eval.skill_eval.trace.id,eval.skill_eval.trace.timestamp
0,241fd291-5bb2-4b28-870e-6cc04ca6ba56,"Amanda offered Jerry some cookies she baked, a...",None,14f918a90956c44c56201e1388d03668,1782911853060,1,good,The generated summary accurately reflects the ...,9854f96c408f6d04275f555fcb6abc5b,1782912022656
1,ee4883bf-b95e-4611-9e9f-63db42de45d1,Olivia and Oliver discuss their voting prefere...,None,8620e07b8c9b5dc60aa7ba3be5482616,1782911854792,0,bad,The generated summary accurately reflects the ...,9f9fb5f95734dfdfd6e61f3dd79b2258,1782912024998
2,c0e644e2-245d-4f11-8c27-767e02a64806,Kim tells Tim she's in a bad mood because she ...,None,367405280e581b8a6f61c381655e5259,1782911856053,0,bad,The Generated Summary provides more detail tha...,a08602f59342046a66c6b20450d01ba9,1782912027190
3,c35f0687-3bc8-4083-bc58-582d57bfcc10,Edward confides to Rachel that he has feelings...,None,d432992f604c724cb330e68dd5521073,1782911857675,0,bad,The Generated Summary introduces the idea that...,7c0002e64138342fda4a8c46cd56275e,1782912029179
4,81372906-5f44-4807-bf15-4f1afbf90fa7,Sam tells Naomi he overheard Rick saying he’s ...,None,3bf342d54849441b3639e10a0b33fb40,1782911859352,1,good,The Generated Summary accurately reflects the ...,b68effdcbfa589a0aad336f3c794d48e,1782912031316


In [10]:
filtered_v0 = collect_feedback(eval_v0)
filtered_v0[["output", "eval.skill_eval.label", "eval.skill_eval.explanation"]]

,output,eval.skill_eval.label,eval.skill_eval.explanation
1,Olivia and Oliver discuss their voting prefere...,bad,The generated summary accurately reflects the ...
2,Kim tells Tim she's in a bad mood because she ...,bad,The Generated Summary provides more detail tha...
3,Edward confides to Rachel that he has feelings...,bad,The Generated Summary introduces the idea that...
0,"Amanda offered Jerry some cookies she baked, a...",good,The generated summary accurately reflects the ...
4,Sam tells Naomi he overheard Rick saying he’s ...,good,The Generated Summary accurately reflects the ...
5,Neville asks for help remembering his wedding ...,good,The Generated Summary accurately reflects the ...
9,"Matt asked Agnes out on a date, and after some...",good,The Generated Summary accurately reflects the ...
10,"Demi tells Lucas she was promoted, and they ag...",good,The generated summary accurately reflects the ...


In [11]:
from pathlib import Path

def snapshot_skill(version_idx: int) -> str:
    Path(VERSIONS_DIR).mkdir(parents=True, exist_ok=True)
    dest = f"{VERSIONS_DIR}/v{version_idx}.md"
    with open(dest, "w") as f:
        f.write(read_skill())
    return dest

def apply_skill(new_text: str):
    with open(SKILL_PATH, "w") as f:
        f.write(new_text)

In [12]:
meta_prompt = """You are an expert at optimizing Claude "skills" — structured Markdown instruction
files (SKILL.md). You are given the CURRENT SKILL.md and PERFORMANCE DATA from running that skill as
a summarization system prompt over a dialogue dataset. Each record has the model's `output` summary,
an LLM-judge `label` (good/bad) comparing it to a gold reference summary, and an `explanation` of why.

Rewrite the SKILL.md so future summaries earn "good".

CURRENT SKILL.md
================
{current_skill}
================

PERFORMANCE DATA (bad records show failures, good records show what the judge rewards)
================
{feedback_samples}
================

REQUIREMENTS
1. Read every explanation. Find the recurring failure pattern across the `bad` records and contrast
   with the `good` ones.
2. Translate those patterns into GENERAL, REUSABLE instructions — never fixes overfit to specific
   dialogues in the data.
3. Preserve the section structure exactly: Description, Goal, Input, Procedure, Output Format, Rules,
   Quality Checklist. Keep the same Markdown headers.
4. Output ONLY the complete rewritten SKILL.md Markdown. No preamble, no code fences, no commentary.
"""

def _strip_code_fences(text: str) -> str:
    """Defensive: if the model wraps the rewritten SKILL.md in a ``` fence despite the
    instructions, remove it. apply_skill (Task 6) writes this text straight to SKILL.md,
    so a stray fence would silently corrupt the live skill file."""
    t = text.strip()
    if t.startswith("```"):
        lines = t.split("\n")
        lines = lines[1:]  # drop opening fence line (``` or ```markdown)
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]  # drop closing fence line
        t = "\n".join(lines).strip()
    return t

def optimize_skill(current_skill: str, filtered_df) -> str:
    feedback = filtered_df[["output", "eval.skill_eval.label", "eval.skill_eval.explanation"]].to_string(index=False)
    raw = gen_chain.invoke(meta_prompt.format(current_skill=current_skill, feedback_samples=feedback))
    return _strip_code_fences(raw)

In [13]:
snap = snapshot_skill(0)
new_skill_text = optimize_skill(read_skill(), filtered_v0)
print("snapshotted:", snap)
print("---- proposed SKILL.md ----")
print(new_skill_text)
# structural check
for header in ["## Description", "## Goal", "## Procedure", "## Rules", "## Quality Checklist"]:
    assert header in new_skill_text, f"missing section: {header}"
print("\nOK: all required sections present")

snapshotted: dialog-summary/versions/v0.md
---- proposed SKILL.md ----
# Dialogue Summary Skill

## Description

Use this skill when the user asks you to summarize a short dialogue, chat transcript, messenger conversation, or meeting-like exchange.

## Goal

Create a concise summary that captures the main point of the conversation, including important decisions, requests, plans, or emotional context when relevant.

## Input

The input is usually a speaker-labeled dialogue. Each line may contain a speaker name followed by their message.

Example:

Alice: Are you coming to dinner tonight?  
Bob: I might be late because of work.  
Alice: That's okay. I'll save you some food.

## Procedure

1. Read the entire dialogue carefully before summarizing.
2. Identify the main participants and ensure their names are copied exactly as written in the dialogue.
3. Identify the main topic and purpose of the conversation.
4. Extract only the most important decisions, requests, plans, problems, or commit

## Task 6: Iteration loop + improvement report

Iterate `N_ITERATIONS` times: collect feedback from the last experiment, rewrite `SKILL.md`
with the meta-optimizer, snapshot the pre-edit version to `dialog-summary/versions/`, apply the
new skill, and re-run the experiment as `skill-v{i}`. Finally, report the mean score per version
so the v0→vN improvement is visible (and comparable in the Arize UI).

In [14]:
history = [{"version": "skill-v0", "score": mean_score(eval_v0)}]
last_eval = eval_v0

for i in range(1, N_ITERATIONS + 1):
    filtered = collect_feedback(last_eval)
    new_text = optimize_skill(read_skill(), filtered)
    snapshot_skill(i - 1)          # save the version we are about to replace
    apply_skill(new_text)          # SKILL.md now = optimized version i
    exp, last_eval = run_experiment(f"skill-v{i}")
    score = mean_score(last_eval)
    history.append({"version": f"skill-v{i}", "score": score})
    print(f"skill-v{i} mean score: {score:.3f}")

pd.DataFrame(history)

/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  arize.experiments.functions | INFO | 🧪 Experiment started.


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


...

running experiment evaluations |██████████| 100/100 (100.0%) | ⏳ 03:31<00:00 |  2.12s/it

  arize.experiments.functions | INFO | ✅ All evaluators completed.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


skill-v3 mean score: 0.250


,version,score
0,skill-v0,0.29
1,skill-v1,0.22
2,skill-v2,0.23
3,skill-v3,0.25


In [15]:
hist_df = pd.DataFrame(history)
best = hist_df.loc[hist_df["score"].idxmax()]
print(hist_df.to_string(index=False))
print(f"\nBaseline (skill-v0): {history[0]['score']:.3f}")
print(f"Best: {best['version']} @ {best['score']:.3f}")
print("Compare experiments skill-v0..skill-v{} in the Arize UI.".format(N_ITERATIONS))

 version  score
skill-v0   0.29
skill-v1   0.22
skill-v2   0.23
skill-v3   0.25

Baseline (skill-v0): 0.290
Best: skill-v0 @ 0.290
Compare experiments skill-v0..skill-v3 in the Arize UI.
